# 03 — P–T Profile Gallery

What atmospheric profiles does the emulator support? Two cells of context:

1. **Analytic profile shapes used in the literature.** Line+2013 three-channel
   radiative-equilibrium and Mollière+2019 modified-Guillot (the petitRADTRANS
   default). Both are implemented from the original equations and sampled
   from their published prior ranges.
2. **The actual training distribution.** Pulled from the test split of the
   bundle this notebook is pointed at, classified into PT-library / analytic
   radiative / analytic convective buckets.

This notebook does not load the chemistry forward pass — it is purely a
diagnostic for the temperature–pressure inputs.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Resolve the project root whether this notebook is launched from
# `exojax_demo/` or from the repo root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Pick the bundle to load. Override via the VULCAN_DEMO_MODEL env var
# (used by the PBS submission script in supercomputer_cmds/).
import os
MODEL = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

# Put the distribution root on sys.path so `src` imports resolve.
DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

# Project-shipped matplotlib style (ships in this folder).
_STYLE = Path("science.mplstyle")
if _STYLE.exists():
    plt.style.use(str(_STYLE))

print(f"MODEL       : {MODEL}")
print(f"BUNDLE_PATH : {BUNDLE_PATH}")


## Line et al. (2013) profile

Three-channel radiative-equilibrium profile.

$$T^4(\tau) = \frac{3T_{\rm int}^4}{4}\left(\frac{2}{3} + \tau\right)
            + \frac{3T_{\rm irr}^4}{4}\left[(1-\alpha)\xi_1 + \alpha\xi_2\right]$$

$$\tau(P) = \frac{\kappa_{\rm IR}\,P_{\rm ref}}{g\,n}\,P^n
\qquad
\xi_i(\gamma_i,\tau) = \frac{2}{3} + \frac{2}{3\gamma_i}\!\left[1 + \left(\frac{\gamma_i\tau}{2}-1\right)e^{-\gamma_i\tau}\right] + \frac{2\gamma_i}{3}\!\left(1-\frac{\tau^2}{2}\right)E_2(\gamma_i\tau)$$


In [ ]:
from scipy.special import expn

_BAR_TO_PA = 1.0e5


def compute_optical_depth(pressure_bar, kappa_ir_m2_kg, gravity_m_s2, power_law_n):
    tau_scale = (kappa_ir_m2_kg * _BAR_TO_PA) / gravity_m_s2
    return (tau_scale / power_law_n) * pressure_bar ** power_law_n


def compute_xi(gamma, tau):
    gt = gamma * tau
    return (
        2.0 / 3.0
        + (2.0 / (3.0 * gamma)) * (1.0 + (gt / 2.0 - 1.0) * np.exp(-gt))
        + (2.0 * gamma / 3.0) * (1.0 - 0.5 * tau ** 2) * expn(2, gt)
    )


def apply_convective_adjustment(pressure_bar, temperature_k, adiabatic_gradient):
    """Splice on a dry adiabat at the deepest superadiabatic layer."""
    adjusted = np.copy(temperature_k)
    log_p = np.log10(pressure_bar)
    log_t = np.log10(temperature_k)
    for i in range(1, len(adjusted)):
        if (log_t[i] - log_t[i - 1]) / (log_p[i] - log_p[i - 1]) > adiabatic_gradient:
            adjusted[i:] = adjusted[i - 1] * (pressure_bar[i:] / pressure_bar[i - 1]) ** adiabatic_gradient
            break
    return adjusted


def line2013_profile(pressure_bar, t_int_k, t_irr_k,
                     kappa_ir_m2_kg, power_law_n,
                     gamma_1, gamma_2, alpha,
                     gravity_m_s2=25.0,
                     adiabatic_gradient=None,
                     temperature_shift_k=0.0):
    tau = compute_optical_depth(pressure_bar, kappa_ir_m2_kg, gravity_m_s2, power_law_n)
    xi1 = compute_xi(gamma_1, tau)
    xi2 = compute_xi(gamma_2, tau)
    t4 = (3 * t_int_k ** 4 / 4) * (2.0 / 3.0 + tau) + (3 * t_irr_k ** 4 / 4) * (
        (1.0 - alpha) * xi1 + alpha * xi2
    )
    T = np.clip(t4, 0.0, None) ** 0.25
    if adiabatic_gradient is not None:
        T = apply_convective_adjustment(pressure_bar, T, adiabatic_gradient)
    return T + temperature_shift_k


## Mollière+2019 (petitRADTRANS) profile

Guillot (2010) base profile, modified at high altitudes and boxcar-smoothed
in log-pressure (Mollière+2019, Eqs. 15–16):

$$T_{\rm Guillot}^4(P) = \frac{3T_{\rm int}^4}{4}\!\left(\frac{2}{3}+\delta P\right) + \frac{3T_{\rm eq}^4}{4}\!\left[\frac{2}{3}+\frac{1}{\gamma\sqrt{3}}+\left(\frac{\gamma}{\sqrt{3}}-\frac{1}{\gamma\sqrt{3}}\right)e^{-\gamma\delta\sqrt{3}\,P}\right]$$

$$T(P) = \left\langle T_{\rm Guillot}(P)\cdot\left(1 - \frac{\alpha}{1 + P/P_{\rm trans}}\right)\right\rangle_{\!\Delta\log P\,=\,1.25\,\rm dex}$$


In [ ]:
from scipy.ndimage import uniform_filter1d


def _guillot(pressure_bar, delta, gamma, T_int, T_eq):
    tau = delta * pressure_bar
    T4 = (
        3.0 / 4.0 * T_int ** 4 * (2.0 / 3.0 + tau)
        + 3.0 / 4.0 * T_eq ** 4 * (
            2.0 / 3.0
            + 1.0 / (gamma * np.sqrt(3.0))
            + (gamma / np.sqrt(3.0) - 1.0 / (gamma * np.sqrt(3.0)))
            * np.exp(-gamma * np.sqrt(3.0) * tau)
        )
    )
    return np.clip(T4, 0.0, None) ** 0.25


def _boxcar_logP(pressure_bar, T, width_dex=1.25):
    """Running mean over a fixed window in log10(P)."""
    dp = abs(np.log10(pressure_bar[1]) - np.log10(pressure_bar[0]))
    n = max(1, int(round(width_dex / dp)))
    return uniform_filter1d(T, size=n, mode="nearest")


def molliere2019_pt_profile(pressure_bar, log_delta, log_gamma, T_int, T_eq, alpha, log_P_trans):
    delta = 10.0 ** log_delta
    gamma = 10.0 ** log_gamma
    P_trans = 10.0 ** log_P_trans

    T_g = _guillot(pressure_bar, delta, gamma, T_int, T_eq)
    T_mod = T_g * (1.0 - alpha / (1.0 + pressure_bar / P_trans))
    return _boxcar_logP(pressure_bar, T_mod)


## Sample N profiles from each shape

Both branches use rejection sampling with `T ∈ [0, 3000] K` to mirror the
shipped 3000 K cap (VULCAN's reaction networks are validated only to ~3000 K).


In [ ]:
P_GRID = np.logspace(2.0, -7.0, 64)
N_PROFILES = 10
rng = np.random.default_rng()


def sample_line2013(rng):
    return line2013_profile(
        P_GRID,
        t_int_k=rng.normal(500.0, 20.0),
        t_irr_k=np.clip(rng.normal(1800.0, 500.0), 300.0, 4000.0),
        kappa_ir_m2_kg=10.0 ** rng.normal(-2.5, 2.5),
        power_law_n=rng.uniform(0.5, 2.0),
        gamma_1=10.0 ** rng.uniform(-2.0, 2.0),
        gamma_2=10.0 ** rng.uniform(-2.0, 2.0),
        alpha=rng.uniform(0.0, 1.0),
        gravity_m_s2=rng.uniform(5.0, 50.0),
        adiabatic_gradient=rng.uniform(0.25, 0.35) if rng.random() < 1.0 / 3.0 else None,
        temperature_shift_k=rng.uniform(-500.0, 500.0),
    )


def sample_molliere2019(rng):
    log_kappa = rng.normal(-2.5, 2.5)
    log_g = np.log10(rng.uniform(5.0, 50.0))
    T = molliere2019_pt_profile(
        P_GRID,
        log_delta=log_kappa + 5.0 - log_g,
        log_gamma=rng.uniform(-2.0, 2.0),
        T_int=rng.normal(500.0, 20.0),
        T_eq=np.clip(rng.normal(1800.0, 500.0), 300.0, 4000.0),
        alpha=rng.uniform(0.0, 1.0),
        log_P_trans=rng.uniform(-5.0, 1.0),
    )
    if rng.random() < 1.0 / 3.0:
        T = apply_convective_adjustment(P_GRID, T, rng.uniform(0.25, 0.35))
    return T


profiles_line = []
while len(profiles_line) < N_PROFILES:
    T = sample_line2013(rng)
    if T.min() >= 0.0 and T.max() <= 3000.0:
        profiles_line.append(T)

profiles_prt = []
while len(profiles_prt) < N_PROFILES:
    T = sample_molliere2019(rng)
    if T.min() >= 0.0 and T.max() <= 3000.0:
        profiles_prt.append(T)

print(f"sampled {len(profiles_line)} Line+2013 and {len(profiles_prt)} Mollière+2019 profiles")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
fig.subplots_adjust(wspace=0.05)

palette = plt.cm.tab20(np.linspace(0.0, 1.0, N_PROFILES))
for i, T in enumerate(profiles_line):
    ax1.plot(T, P_GRID, lw=2.0, color=palette[i])
ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_xlabel("Temperature (K)")
ax1.set_ylabel("Pressure (bar)")
ax1.set_title("Line+2013")
ax1.grid(True, alpha=0.3)

for i, T in enumerate(profiles_prt):
    ax2.plot(T, P_GRID, lw=2.0, color=palette[i])
ax2.set_yscale("log")
ax2.set_xlabel("Temperature (K)")
ax2.set_title("Mollière+2019 (petitRADTRANS)")
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelleft=False)

plt.tight_layout()
plt.show()


## Training distribution: bundle's test-set profiles

These come from the actual test split of the model bundle. The shape buckets
match `src/data_generation/sampling.py`:

- **PT-library** — imported from a curated PT library entry.
- **Analytic radiative** — modified Guillot, no convective adjustment.
- **Analytic convective** — Guillot with the dry-adiabat splice applied.


In [ ]:
from src.models.standalone_inference import load_model
from src.models.classical_reference import (
    classify_temperature_profile_bucket,
    load_fastchem_raw_metadata_map,
    load_fastchem_test_case,
    load_fastchem_test_context,
)

model = load_model(BUNDLE_PATH)
ctx = load_fastchem_test_context(
    BUNDLE_PATH, model.config, project_root=PROJECT_ROOT, require_raw=True,
)
metadata_map = load_fastchem_raw_metadata_map(ctx.raw_root, ctx.split.run_ids)

PROFILES_PER_BUCKET = 5
BUCKET_STYLE = {
    "pt_library": dict(cmap="Reds", ls="-", lw=2.0, label="PT-library"),
    "analytic_radiative": dict(cmap="Blues", ls="--", lw=2.0, label="Analytic (radiative)"),
    "analytic_convective": dict(cmap="Purples", ls=(0, (5, 2, 1, 2)), lw=2.2, label="Analytic (convective)"),
}

buckets: dict[str, list[str]] = {name: [] for name in BUCKET_STYLE}
for rid in ctx.split.run_ids:
    buckets[classify_temperature_profile_bucket(metadata_map[rid])].append(rid)

bucket_rng = np.random.default_rng()
fig, ax = plt.subplots(figsize=(8, 8))
for bucket_name, rids in buckets.items():
    style = BUCKET_STYLE[bucket_name]
    n_pick = min(PROFILES_PER_BUCKET, len(rids))
    if not n_pick:
        continue
    picked = [str(r) for r in bucket_rng.choice(rids, size=n_pick, replace=False)]
    cases = [load_fastchem_test_case(ctx, rid) for rid in picked]
    cmap = plt.get_cmap(style["cmap"])
    palette = [cmap(0.35 + 0.55 * i / max(len(cases) - 1, 1)) for i in range(len(cases))]
    for i, case in enumerate(cases):
        ax.plot(case.temperature_k, case.pressure_bar,
                ls=style["ls"], lw=style["lw"], alpha=0.85,
                color=palette[i],
                label=style["label"] if i == 0 else None)

ax.set_yscale("log")
ax.set_ylim(1.0e2, 1.0e-7)
ax.set_xlim(0.0, 4000.0)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Pressure (bar)")
ax.set_title(f"Test-set P–T profiles by bucket ({MODEL})")
ax.legend(loc="best")
plt.tight_layout()
plt.show()
